In [1]:
import pickle
import numpy as np
import pandas as pd
import xgboost as xgb
import random

def norm_f(m, a=4443.76, b=1.53, c=4443.76, d=0):
    return (a / ((m**b) + c)) + d

def calcular_score(df_resultados, alpha_peso=2.0):
    
    y_true = df_resultados['RUL_Real'].values
    y_pred = df_resultados['RUL_Final'].values

    resultados_viagem = []

    for porta_id, df_porta in df_resultados.groupby('Porta_ID'):
        
        # T: Número total de pontos preditos nessa viagem
        T = len(df_porta)
        
        # Prevenção contra divisão por zero em viagens vazias
        if T == 0:
            continue
        
        # Ordena o tempo (Fundamental para o PH funcionar)
        df_porta = df_porta.sort_values('Ciclo_Atual')
        
        erro_t = df_porta['RUL_Real'] - df_porta['RUL_Final'] 
        
        # ---------------------------------------
        # SPREAD OF ERROR (SDE)
        # ---------------------------------------
        erro_medio = erro_t.mean() 
        diferencas_quadradas = (erro_t - erro_medio) ** 2 
        somatorio = np.sum(diferencas_quadradas)
        metric_val = np.sqrt(somatorio / T)

        # ---------------------------------------
        # RMSE
        # ---------------------------------------
        erro_quadrado = erro_t ** 2 
        somatorio = np.sum(erro_quadrado)
        mse = somatorio / T
        rmse_val = np.sqrt(mse)

        # ---------------------------------------
        # NORMALIZAÇÃO IMEDIATA
        # ---------------------------------------
        sde_norm = norm_f(metric_val)
        rmse_norm = norm_f(rmse_val)
        
        # ---------------------------------------
        # ACCURACY METRIC
        # ---------------------------------------
        erro_absoluto = np.abs(df_porta['RUL_Real'] - df_porta['RUL_Final']) 
        rul_real = np.maximum(df_porta['RUL_Real'], 1e-5) 
        expoente = -(erro_absoluto / rul_real) 
        valores_exponenciais = np.exp(expoente) 
        acc_val = valores_exponenciais.mean() 

        # ---------------------------------------
        # PROGNOSTIC HORIZON (PH INDIVIDUAL)
        # ---------------------------------------
        t_eof = df_porta['Ciclo_Atual'].max() 
        
        margem_constante = t_eof * 0.10
        limite_inferior = df_porta['RUL_Real'] - margem_constante
        limite_superior = df_porta['RUL_Real'] + margem_constante
        
        dentro_dos_limites = (df_porta['RUL_Final'] >= limite_inferior) & (df_porta['RUL_Final'] <= limite_superior)
        fora_dos_limites = ~dentro_dos_limites 
        
        if fora_dos_limites.any():
            ultimo_erro = df_porta.loc[fora_dos_limites, 'Ciclo_Atual'].max()
            df_reta_final = df_porta[df_porta['Ciclo_Atual'] > ultimo_erro]
            
            if df_reta_final.empty:
                t_alpha = t_eof 
            else:
                t_alpha = df_reta_final['Ciclo_Atual'].min()
        else:
            t_alpha = df_porta['Ciclo_Atual'].min()
                
        ph = (t_eof - t_alpha) / t_eof if t_eof > 0 else 0

        # ---------------------------------------
        # SCORE 
        # ---------------------------------------
        score = (rmse_norm + sde_norm + (alpha_peso * ph)) / (2 + alpha_peso)


        resultados_viagem.append({
            'Porta_ID': porta_id,
            'T_Pontos': T,
            'Erro_Medio (ε_barra)': round(erro_medio, 2),
            'SDE_Bruto': round(metric_val, 2),
            'SDE_Norm': round(sde_norm, 4),
            'RMSE_Bruto': round(rmse_val, 2),
            'RMSE_Norm': round(rmse_norm, 4),
            'Accuracy_Exp': round(acc_val, 4),
            'PH': round(ph, 4),
            'Score': round(score, 4)
        })

    # Converte a lista de resultados no DataFrame de análise detalhada
    df_metricas = pd.DataFrame(resultados_viagem)

    print("\n--- Métricas Individuais por Porta ---")
    print(df_metricas.to_string(index=False))


# ========================================================
# 1. CARREGAMENTO DOS DADOS 
# ========================================================
with open('Dados_Processados/todas_as_portas_features.pkl', 'rb') as f:
    todas_as_portas = pickle.load(f)

random.seed(42)
todos_ids = [porta_id for porta_id, _ in todas_as_portas]
qtd_treino = int(0.9 * len(todos_ids))
ids_treino = random.sample(todos_ids, qtd_treino)
ids_teste = [pid for pid in todos_ids if pid not in ids_treino]

# ========================================================
# 2. JANELA DESLIZANTE (CLIP RUL)
# ========================================================
janela_obs = 50        
passo_deslizamento = 5 

X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []

print("Gerando dados com as Novas Features Avançadas (Sinais Vitais)...")

for porta_id, df in todas_as_portas:
    vida_total = df['Ciclo_Relativo'].max()
    if vida_total < janela_obs: continue
        
    df = df.sort_values('Ciclo_Relativo').reset_index(drop=True)
    
    for inicio in range(0, int(vida_total - janela_obs + 1), passo_deslizamento):
        fim = inicio + janela_obs
        df_janela = df[(df['Ciclo_Relativo'] >= inicio) & (df['Ciclo_Relativo'] < fim)]
        
        if df_janela.empty or len(df_janela) < 4: continue 
            
        sintomas = {
            'Porta_ID': porta_id, 
            'Ciclo_Atual': fim, 
            'Distancia_Ref_Atual': df_janela['Distancia_Ref'].iloc[-1],
            'Inclinacao_Mecanica': df_janela['Distancia_Ref'].iloc[-1] - df_janela['Distancia_Ref'].iloc[0],
            
            'Corrente_RMS_Max': df_janela['Corrente_RMS'].max(),
            'Corrente_RMS_Media': df_janela['Corrente_RMS'].mean(),
            'Tendencia_Corrente': df_janela['Corrente_RMS'].iloc[-1] - df_janela['Corrente_RMS'].iloc[0],
            'Corrente_RMS_Std': df_janela['Corrente_RMS'].std(),                  
            'Corrente_RMS_Kurtosis': df_janela['Corrente_RMS'].kurtosis(),        
            'Corrente_Energia': (df_janela['Corrente_RMS']**2).sum(),             
            
            'Tensao_RMS_Media': df_janela['Tensao_RMS'].mean(),
            'Tendencia_Tensao': df_janela['Tensao_RMS'].iloc[-1] - df_janela['Tensao_RMS'].iloc[0],
            'Potencia_Aparente': df_janela['Tensao_RMS'].mean() * df_janela['Corrente_RMS'].mean()
        }
        
        rul_restante = vida_total - fim
        
        if porta_id in ids_treino:
            X_train_list.append(sintomas)
            y_train_list.append(rul_restante)
        else:
            X_test_list.append(sintomas)
            y_test_list.append(rul_restante)

X_train = pd.DataFrame(X_train_list).fillna(0) 
y_train_bruto = np.array(y_train_list)
X_test = pd.DataFrame(X_test_list).fillna(0)
y_test_bruto = np.array(y_test_list)

rul_maximo = 400
y_train = np.clip(y_train_bruto, a_min=None, a_max=rul_maximo)
y_test = np.clip(y_test_bruto, a_min=None, a_max=rul_maximo)

X_train_model = X_train.drop(columns=['Porta_ID'])
X_test_model = X_test.drop(columns=['Porta_ID'])

# ========================================================
# 3. TREINAMENTO
# ========================================================
print("Treinando o XGBoost Campeão com Lentes de Alta Resolução...")
modelo_xgb_campeao = xgb.XGBRegressor(
    n_estimators=300, 
    max_depth=4, 
    learning_rate=0.05, 
    subsample=0.9,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
modelo_xgb_campeao.fit(X_train_model, y_train)

# ========================================================
# 4. APLICAÇÃO DOS FILTROS INDUSTRIAIS (EMA + MONOTÔNICO)
# ========================================================
print("Aplicando Filtros (EMA + Monotônico)...")
previsoes_brutas = modelo_xgb_campeao.predict(X_test_model)

previsoes_brutas = np.where(previsoes_brutas > 360, 400, previsoes_brutas)

# O SEGREDO AQUI: Adicionamos o 'RUL_Real_Bruto' (sem o corte de 400) para a tabela
df_resultados = pd.DataFrame({
    'Porta_ID': X_test['Porta_ID'],
    'Ciclo_Atual': X_test['Ciclo_Atual'],
    'RUL_Real': y_test,               # Usado para o cálculo do Score Oficial
    'RUL_Real_Bruto': y_test_bruto,   # <--- NOVO: Usado APENAS para o Print da Tabela
    'RUL_Previsto_Bruto': previsoes_brutas,
})

df_resultados = df_resultados.sort_values(['Porta_ID', 'Ciclo_Atual'])

df_resultados['RUL_Suavizado'] = df_resultados.groupby('Porta_ID')['RUL_Previsto_Bruto'].transform(
    lambda x: x.ewm(span=3, adjust=False).mean()
)

df_resultados['RUL_Final'] = df_resultados.groupby('Porta_ID')['RUL_Suavizado'].cummin()

calcular_score(df_resultados)

# ========================================================
# 5. GERAÇÃO DA TABELA VISUAL COMPLETA COM SCROLL (PLOTLY)
# ========================================================
import plotly.graph_objects as go

print(f"\nGerando tabelas visuais interativas no navegador (Viagem Completa)...\n")

# Mantive o [:4] para abrir apenas 4 abas no navegador. Remova para ver todas.
for porta_escolhida in ids_teste[:4]: 
    df_porta = df_resultados[df_resultados['Porta_ID'] == porta_escolhida].copy().sort_values('Ciclo_Atual')
    
    if df_porta.empty:
        continue
        
    # 1. TEMPO E RUL
    df_porta['t'] = df_porta['Ciclo_Atual'] - df_porta['Ciclo_Atual'].min()
    df_porta['RUL real'] = df_porta['RUL_Real_Bruto'] 
    df_porta['RUL predito'] = df_porta['RUL_Final'].round(2)
    
    # 2. CÁLCULO DA MARGEM
    t_eol_viagem = df_porta['RUL real'].iloc[0]
    alpha_val = t_eol_viagem * 0.10
    
    # O .round(1) para cravar apenas uma casa decimal e evitar o N.99999
    df_porta['+alpha'] = (df_porta['RUL real'] + alpha_val).round(1)
    df_porta['-alpha'] = (df_porta['RUL real'] - alpha_val).round(1)
    
    df_porta = df_porta.reset_index(drop=True)
    
    # 3. ENCONTRA O T_ALPHA
    esta_na_banda = (df_porta['RUL predito'] >= df_porta['-alpha']) & (df_porta['RUL predito'] <= df_porta['+alpha'])
    
    if esta_na_banda.any():
        idx_acerto = esta_na_banda.idxmax()
        t_alpha_val = df_porta.loc[idx_acerto, 't']
        status_texto = f"<span style='color: #27ae60;'>Acerto no t={t_alpha_val}</span>"
    else:
        idx_acerto = float('inf') # Define como infinito se nunca acertar
        t_alpha_val = "NaN"
        status_texto = "<span style='color: #c0392b;'>Nunca entrou na margem</span>"

    # ==========================================
    # LÓGICA DAS CORES DAS LINHAS
    # ==========================================
    cores_linhas = []
    for i in range(len(df_porta)):
        if i >= idx_acerto:
            # Verde claro a partir do t_alpha (com leve zebrado para leitura)
            cores_linhas.append('#e8f5e9' if i % 2 == 0 else '#d4edda') 
        else:
            # Zebrado azul/branco padrão antes do acerto
            cores_linhas.append('white' if i % 2 == 0 else '#f2f7f9')

    # 4. REMOVE A COLUNA T_ALPHA DA TABELA FINAL
    tabela_final = df_porta[['t', 'RUL real', 'RUL predito', '+alpha', '-alpha']]
    
    # O Plotly exige que a cor seja passada para cada coluna, então multiplicamos a lista de cores
    matriz_cores = [cores_linhas] * len(tabela_final.columns)

    # ==========================================
    # CRIANDO A TABELA COM SCROLL NO PLOTLY
    # ==========================================
    titulo_tabela = (
        f"<b>DIAGNÓSTICO DA VIAGEM COMPLETA - PORTA {porta_escolhida}</b><br>"
        f"<sup style='font-size: 14px;'>t_EoL: {t_eol_viagem:.0f} ciclos | Margem (&alpha;): &plusmn;{alpha_val:.1f} | t_alpha: {status_texto}</sup>"
    )

    fig = go.Figure(data=[go.Table(
        header=dict(
            values=[f"<b>{col}</b>" for col in tabela_final.columns],
            line_color='darkslategray',
            fill_color='#2c3e50', 
            align='center',
            font=dict(color='white', size=14),
            height=40
        ),
        cells=dict(
            values=[tabela_final[col] for col in tabela_final.columns],
            line_color='darkslategray',
            fill_color=matriz_cores, # <--- Aplica a nossa lógica de cores dinâmicas!
            align='center',
            font=dict(color='darkslategray', size=13),
            height=30
        )
    )])

    fig.update_layout(
        title_text=titulo_tabela,
        title_x=0.5, 
        margin=dict(l=20, r=20, t=80, b=20),
        width=900,
        height=600 
    )

    fig.show(renderer='browser')


# ========================================================
# 6. EXPORTAÇÃO DO ARQUIVO DE SUBMISSÃO
# ========================================================
print("\nGerando arquivo de submissão...")

# 1. Cria um dataframe novo só com as colunas que importam
df_submissao = df_resultados[['Porta_ID', 'RUL_Final']].copy()

# 2. Arredonda o RUL para ficar com números inteiros ou 1 casa decimal (conforme a imagem)
# A imagem mostra números inteiros, então vamos arredondar e converter para int
df_submissao['RUL_Final'] = df_submissao['RUL_Final'].round(0).astype(int)

# 3. Renomeia as colunas para não ter problema no sistema deles (opcional, mas recomendado)
df_submissao.columns = ['Test_ID', 'RUL']

# 4. Salva no formato CSV (sem salvar a coluna de índice do Pandas)
nome_arquivo = 'submission_xgboost_phm.csv'
df_submissao.to_csv(nome_arquivo, index=False, header=False) # header=False se eles não quiserem o nome das colunas no arquivo

print(f"✅ Arquivo '{nome_arquivo}' gerado com sucesso!")
print("Prévia das 5 primeiras e 5 últimas linhas:")
print(df_submissao.head())
print("...")
print(df_submissao.tail())

Gerando dados com as Novas Features Avançadas (Sinais Vitais)...
Treinando o XGBoost Campeão com Lentes de Alta Resolução...
Aplicando Filtros (EMA + Monotônico)...

--- Métricas Individuais por Porta ---
 Porta_ID  T_Pontos  Erro_Medio (ε_barra)  SDE_Bruto  SDE_Norm  RMSE_Bruto  RMSE_Norm  Accuracy_Exp     PH  Score
       11       308                  3.46      16.85    0.9833       17.20     0.9828        0.9578 0.9685 0.9758
       22       101                  7.61      31.54    0.9577       32.45     0.9559        0.8657 0.2545 0.6057
       24        35                -25.43      21.26    0.9764       33.15     0.9545        0.6346 0.0682 0.5168
       25       419                 -3.19      22.95    0.9735       23.17     0.9732        0.9394 0.9766 0.9750

Gerando tabelas visuais interativas no navegador (Viagem Completa)...


Gerando arquivo de submissão...
✅ Arquivo 'submission_xgboost_phm.csv' gerado com sucesso!
Prévia das 5 primeiras e 5 últimas linhas:
   Test_ID  RUL
0 